# MN-SENA – interactive projection viewer

Pick a **year** and a **projection**, then click any point to see its q-transform PDF.

1. Run the *compute* cell once. It is slow, and it caches the embeddings in `embeddings_cache/`.
2. From then on, only run the *config* cell and the *viewer* cell.

Only `get_qtransform_path` (viewer cell) assumes a PDF folder layout: it is copied from `virgo_year_int.ipynb`. Check it against the MN-SENA folders.

Made with help of claude ai

In [1]:
import os
import re
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
METRIC = "euclidean"

PATH = Path("year_MNSENA_results_cut")      # parquet files + qtransform PDFs
CACHE_DIR = Path("embeddings_cache")        # embeddings are stored here
CACHE_DIR.mkdir(exist_ok=True)

# year -> t-SNE perplexity (same values as mn-sena-splited.ipynb)
YEARS = {2022: 250, 2023: 220, 2024: 220, 2025: 220}

# label shown in the UI -> key stored inside embeddings_<year>.npz
PROJECTIONS = {"t-SNE": "tsne", "UMAP": "umap"}

In [2]:
# Run once. Skips any year that is already cached.
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import umap

for year, perplexity in YEARS.items():
    out = CACHE_DIR / f"embeddings_{year}.npz"
    if out.exists():
        print(f"{year}: cached, skipping")
        continue

    print(f"{year}: computing...")
    df = pd.read_parquet(PATH / f"qtransform_features_{year}.parquet")
    feature_columns = [c for c in df.columns if c.startswith("feature_")]
    X = df[feature_columns].to_numpy(dtype="float64")

    X_norm = X / X.max(axis=1, keepdims=True)
    X_scaled = StandardScaler().fit_transform(X_norm)

    X_tsne = TSNE(n_components=2, perplexity=perplexity, early_exaggeration=14,
                  init="pca", learning_rate="auto",
                  random_state=RANDOM_STATE, metric=METRIC).fit_transform(X_scaled)
    X_umap = umap.UMAP(random_state=RANDOM_STATE).fit_transform(X_scaled)

    # metadata (everything except the features) so the viewer never reloads the big parquet
    df.drop(columns=feature_columns).to_parquet(CACHE_DIR / f"meta_{year}.parquet")
    np.savez_compressed(out, tsne=X_tsne, umap=X_umap)   # saved last = "done" marker
    del df, X, X_norm, X_scaled

2022: computing...


/home/rauls/Desktop/VirgoBRET/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


2023: computing...


/home/rauls/Desktop/VirgoBRET/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


2024: computing...


/home/rauls/Desktop/VirgoBRET/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


2025: computing...


/home/rauls/Desktop/VirgoBRET/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [2]:
import pymupdf
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, Image, clear_output

PDF_DPI = 150
IMAGE_WIDTH = 900


def parse_date_from_filename(filename):
    match = re.search(r"(20\d{2})-(\d{2})-(\d{2})", str(filename))
    if match is None:
        raise ValueError(f"Could not extract date from filename: {filename}")
    y, m, d = match.groups()
    return pd.Timestamp(year=int(y), month=int(m), day=int(d))


state = {}   # year -> dict(df, xy, customdata), loaded on first use


def load_year(year):
    if year in state:
        return state[year]

    df = pd.read_parquet(CACHE_DIR / f"meta_{year}.parquet")

    # label/run lines kept exactly as in virgo_year_int.ipynb
    if "label" not in df.columns:
        df["label"] = "Unlabeled"
    label_map = df.drop_duplicates("file").set_index("file")["label"]
    df["label"] = df["file"].map(label_map).fillna("Unlabeled")
    df["run"] = df["file"].map(label_map).fillna("Unlabeled")

    df["_file_date"] = df["file"].apply(parse_date_from_filename)
    df["_day_str"] = df["_file_date"].dt.strftime("%Y%m%d")
    df["_qtransform_index"] = (df.groupby(["run", "station", "channel", "month_year", "_day_str"], sort=False).cumcount())

    emb = np.load(CACHE_DIR / f"embeddings_{year}.npz")
    state[year] = dict(
        df=df,
        xy={name: emb[key] for name, key in PROJECTIONS.items()},
        customdata=np.column_stack([np.arange(len(df)), df["file"].astype(str).values, df["label"].astype(str).values]))
    return state[year]


def get_qtransform_path(df, df_index):
    row = df.iloc[df_index]
    file = f"qtransform_{row['_day_str']}_{int(row['_qtransform_index']):06d}.pdf"
    return PATH / str(row["year"]) / str(row["station"]) / str(row["channel"]) / str(row["month_year"]) / file


@lru_cache(maxsize=64)
def render_pdf(pdf_path, dpi=PDF_DPI):
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF not found:\n{pdf_path}")
    pdf = pymupdf.open(pdf_path)
    try:
        if len(pdf) == 0:
            raise ValueError("PDF contains no pages.")
        zoom = dpi / 72.0
        pix = pdf[0].get_pixmap(matrix=pymupdf.Matrix(zoom, zoom), alpha=False)
        return pix.tobytes("png")
    finally:
        pdf.close()


# ---------------------------------------------------------------- widgets
year_dd = widgets.Dropdown(options=list(YEARS), value=list(YEARS)[0], description="Year:")
proj_tb = widgets.ToggleButtons(options=list(PROJECTIONS), description="Projection:")
image_output = widgets.Output(layout=widgets.Layout(width="1000px"))

fig = go.FigureWidget(data=[go.Scattergl(
    x=[], y=[], mode="markers", marker=dict(size=5, opacity=0.5),
    hovertemplate=(
        "<b>DataFrame index:</b> %{customdata[0]}<br>"
        "<b>File:</b> %{customdata[1]}<br>"
        "<b>Label:</b> %{customdata[2]}<br>"
        "<b>x:</b> %{x:.3f}<br><b>y:</b> %{y:.3f}<extra></extra>"))])
fig.update_layout(width=1000, height=800, hovermode="closest")


def make_title(year, proj):
    if proj == "t-SNE":
        return (f"{year} | t-SNE | Perplexity = {YEARS[year]} | Early Exaggeration = 14 | "
                f"PCA Components = No PCA | Metric = {METRIC}")
    return f"{year} | {proj}"


def update_plot(*_):
    year, proj = year_dd.value, proj_tb.value
    d = load_year(year)
    xy = d["xy"][proj]
    with fig.batch_update():
        trace = fig.data[0]
        trace.x = xy[:, 0]
        trace.y = xy[:, 1]
        trace.customdata = d["customdata"]
        fig.layout.title = make_title(year, proj)
        fig.layout.xaxis.title = f"{proj} 1"
        fig.layout.yaxis.title = f"{proj} 2"
        fig.layout.xaxis.autorange = True
        fig.layout.yaxis.autorange = True


def on_year_change(change):
    with image_output:
        clear_output()        # the PDF on screen belongs to the previous year
    update_plot()


def show_qtransform(df_index, file, label, xy):
    year = year_dd.value
    pdf_path = get_qtransform_path(state[year]["df"], df_index)
    with image_output:
        clear_output(wait=True)
        print(f"Year            : {year}")
        print(f"DataFrame index : {df_index}")
        print(f"File            : {file}")
        print(f"Label           : {label}")
        print(f"PDF             : {pdf_path}")
        print(f"{proj_tb.value:<16}: ({xy[0]:.4f}, {xy[1]:.4f})\n")
        try:
            display(Image(data=render_pdf(pdf_path), width=IMAGE_WIDTH))
        except FileNotFoundError as e:
            print("ERROR: PDF file was not found.")
            print(e)
        except Exception as e:
            print("ERROR while rendering PDF:")
            print(e)


def on_click(trace, points, selector):
    if not points.point_inds:
        return
    i = points.point_inds[0]
    cd = trace.customdata[i]
    show_qtransform(int(cd[0]), str(cd[1]), str(cd[2]), (trace.x[i], trace.y[i]))


fig.data[0].on_click(on_click)
year_dd.observe(on_year_change, names="value")
proj_tb.observe(update_plot, names="value")

update_plot()
display(widgets.HBox([year_dd, proj_tb]))
display(fig)
display(image_output)

FigureWidget({
    'data': [{'customdata': array([[0, 'SENA_HHZ_2022-01-04.mseed', 'Unlabeled'],
                                   [1, 'SENA_HHZ_2022-01-04.mseed', 'Unlabeled'],
                                   [2, 'SENA_HHZ_2022-01-05.mseed', 'Unlabeled'],
                                   ...,
                                   [57509, 'SENA_HHZ_2022-12-31.mseed', 'Unlabeled'],
                                   [57510, 'SENA_HHZ_2022-12-31.mseed', 'Unlabeled'],
                                   [57511, 'SENA_HHZ_2022-12-31.mseed', 'Unlabeled']],
                                  shape=(57512, 3), dtype=object),
              'hovertemplate': ('<b>DataFrame index:</b> %{cust' ... 'y:</b> %{y:.3f}<extra></extra>'),
              'marker': {'opacity': 0.5, 'size': 5},
              'mode': 'markers',
              'type': 'scattergl',
              'uid': '13cd9188-7965-488d-a859-4ffd0f0028a3',
              'x': array([  6.0269184, -20.014738 , -30.233967 , ..., -38.511185 , -30.

Output(layout=Layout(width='1000px'))